In [2]:
from monai.apps.auto3dseg import AutoRunner
import json
import shutil
from pathlib import Path
import os

from helpers.paths import DATA_ROOT

In [ ]:
train_home = Path("/home/srs-9/Projects/prl_project/training/full_brain")
train_root = Path("/media/smbshare/srs-9/prl_project/training/full_brain")
train_name = "swinunetr_test1"
work_dir = train_root / train_name
work_dir.mkdir(exist_ok=True)

datalist_dst = work_dir/"datalist.json"
shutil.copy2(train_home/"datalist.json", datalist_dst)

# input_cfg = {
#     "modality": "mri",
#     "class_names": ["lesion", "rim"],
#     "algos": ["swinunetr"],
#     "work_dir": str(work_dir),
#     "datalist": str(datalist_dst),
#     "dataroot": str(DATA_ROOT),
#     "analyze": True,
#     "algo_gen": True,
#     "train": False,
#     "ensemble": False
# }

input_overrides = "/home/srs-9/Projects/prl_project/training/full_brain/input_overrides.yaml"

runner = AutoRunner(input=input_overrides)
runner.run()

Loading input config /home/srs-9/Projects/prl_project/training/full_brain/input_overrides.yaml
AutoRunner using work directory /media/smbshare/srs-9/prl_project/training/full_brain/swinunetr_test1
Found num_fold 5 based on the input datalist /media/smbshare/srs-9/prl_project/training/full_brain/swinunetr_test1/datalist.json.
Setting num_fold 5 based on the input datalist /media/smbshare/srs-9/prl_project/training/full_brain/swinunetr_test1/datalist.json.
Using user defined command running prefix , will override other settings
Running data analysis...


2026-03-30 19:09:54,068 - INFO - Found 1 GPUs for data analyzing!


  0%|          | 0/20 [00:00<?, ?it/s]

100%|██████████| 20/20 [00:20<00:00,  1.01s/it]

2026-03-30 19:10:14,211 - INFO - Data spacing is not completely uniform. MONAI transforms may provide unexpected result
2026-03-30 19:10:14,211 - INFO - Writing data stats to /media/smbshare/srs-9/prl_project/training/full_brain/swinunetr_test1/datastats.yaml.
2026-03-30 19:10:14,236 - INFO - Writing by-case data stats to /media/smbshare/srs-9/prl_project/training/full_brain/swinunetr_test1/datastats_by_case.yaml, this may take a while.
2026-03-30 19:10:14,348 - INFO - BundleGen from https://github.com/Project-MONAI/research-contributions/releases/download/algo_templates/21ed8e5.tar.gz



algo_templates.tar.gz: 104kB [00:00, 601kB/s]                              

2026-03-30 19:10:14,695 - INFO - Downloaded: /tmp/tmp0kthnadu/algo_templates.tar.gz
2026-03-30 19:10:14,695 - INFO - Expected md5 is None, skip md5 check for file /tmp/tmp0kthnadu/algo_templates.tar.gz.
2026-03-30 19:10:14,697 - INFO - Writing into directory: /media/smbshare/srs-9/prl_project/training/full_brain/swinunetr_test1.


2026-03-30 19:10:16,212 - INFO - Generated:/media/smbshare/srs-9/prl_project/training/full_brain/swinunetr_test1/swinunetr_0
2026-03-30 19:10:16,696 - INFO - Generated:/media/smbshare/srs-9/prl_project/training/full_brain/swinunetr_test1/swinunetr_1
2026-03-30 19:10:17,225 - INFO - Generated:/media/smbshare/srs-9/prl_project/training/full_brain/swinunetr_test1/swinunetr_2
2026-03-30 19:10:17,657 - INFO - Generated:/media/smbshare/srs-9/prl_project/training/full_brain/swinunetr_test1/swinunetr_3
2026-03-30 19:10:18,120 - INFO - Generated:/media/smbshare/srs-9/prl_project/training/full_brain/swinunetr_test1/swinunetr_4


Skipping algorithm training...
Auto3Dseg pipeline is completed successfully.


In [19]:
from monai.bundle import ConfigParser


def patch_swinunetr_configs(work_dir, overrides, num_folds=5, algo_name="swinunetr"):
    """Patch generated config YAMLs with user overrides.

    Uses MONAI's '#' nested-key notation, e.g. "loss#weight" targets config["loss"]["weight"],
    and "transforms_train#transforms#9#ratios" targets the 10th transform's ratios field.
    """
    work_dir = Path(work_dir)
    for fold in range(num_folds):
        config_dir = work_dir / f"{algo_name}_{fold}" / "configs"
        for config_file, patches in overrides.items():
            fpath = config_dir / config_file
            parser = ConfigParser(globals=False)
            parser.read_config(str(fpath))
            for key, value in patches.items():
                parser[key] = value
            ConfigParser.export_config_file(
                parser.get(), str(fpath), fmt="yaml", default_flow_style=None
            )


overrides = {
    "hyper_parameters.yaml": {
        "loss#weight": "$torch.tensor([1.0, 1.0, 10.0]).cuda()",
    },
    "transforms_train.yaml": {
        "transforms_train#transforms#9#ratios": [0, 1, 5],
    },
}
patch_swinunetr_configs(work_dir, overrides)

In [21]:
# Verify overrides took effect
p = ConfigParser(globals=False)
p.read_config(str(work_dir / "swinunetr_0/configs/hyper_parameters.yaml"))
print("loss:", p["loss"])

p2 = ConfigParser(globals=False)
p2.read_config(str(work_dir / "swinunetr_0/configs/transforms_train.yaml"))
crop_transform = p2["transforms_train#transforms#9"]
print("RandCropByLabelClassesd:", crop_transform)

loss: {'_target_': 'DiceCELoss', 'include_background': True, 'sigmoid': '$not @softmax', 'smooth_dr': 1e-05, 'smooth_nr': 0, 'softmax': '$@softmax', 'squared_pred': True, 'to_onehot_y': '$@softmax', 'weight': '$torch.tensor([1.0, 1.0, 10.0]).cuda()'}
RandCropByLabelClassesd: {'_target_': 'RandCropByLabelClassesd', 'keys': ['@image_key', '@label_key'], 'label_key': '@label_key', 'num_classes': '@output_classes', 'num_samples': '@num_crops_per_image', 'ratios': [0, 1, 5], 'spatial_size': '@roi_size', 'warn': False}


In [22]:
# Train with patched configs (skips analyze + algo_gen since bundles already exist)
runner2 = AutoRunner(
    work_dir=str(work_dir),
    input=input_overrides,
    analyze=False,
    algo_gen=False,
    train=True,
    ensemble=True,
)
runner2.run()

Loading input config /home/srs-9/Projects/prl_project/training/full_brain/input_overrides.yaml
AutoRunner using work directory /media/smbshare/srs-9/prl_project/training/full_brain/swinunetr_test1
Found num_fold 5 based on the input datalist /media/smbshare/srs-9/prl_project/training/full_brain/swinunetr_test1/datalist.json.
Setting num_fold 5 based on the input datalist /media/smbshare/srs-9/prl_project/training/full_brain/swinunetr_test1/datalist.json.
Using user defined command running prefix , will override other settings
Skipping data analysis...
Skipping algorithm generation...


2026-03-30 19:42:38,949 - INFO - ['python', '/media/smbshare/srs-9/prl_project/training/full_brain/swinunetr_test1/swinunetr_0/scripts/train.py', 'run', "--config_file='/media/smbshare/srs-9/prl_project/training/full_brain/swinunetr_test1/swinunetr_0/configs/hyper_parameters.yaml,/media/smbshare/srs-9/prl_project/training/full_brain/swinunetr_test1/swinunetr_0/configs/network.yaml,/media/smbshare/srs-9/prl_project/training/full_brain/swinunetr_test1/swinunetr_0/configs/transforms_infer.yaml,/media/smbshare/srs-9/prl_project/training/full_brain/swinunetr_test1/swinunetr_0/configs/transforms_train.yaml,/media/smbshare/srs-9/prl_project/training/full_brain/swinunetr_test1/swinunetr_0/configs/transforms_validate.yaml'"]


<frozen importlib._bootstrap_external>:1324: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.


2026-03-30 19:42:52,450 - INFO - Downloaded: /media/smbshare/srs-9/prl_project/training/full_brain/swinunetr_test1/swinunetr_0/pretrained_model/swin_unetr.base_5000ep_f48_lr2e-4_pretrained.pt
2026-03-30 19:42:52,450 - INFO - Expected md5 is None, skip md5 check for file /media/smbshare/srs-9/prl_project/training/full_brain/swinunetr_test1/swinunetr_0/pretrained_model/swin_unetr.base_5000ep_f48_lr2e-4_pretrained.pt.


The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
2026/03/30 19:42:52 INFO mlflow.tracking.fluent: Experiment with name 'Auto3DSeg' does not exist. Creating a new experiment.
swinunetr_0 - training ...:   0%|          | 0/200 [00:12<?, ?round/s]
Exception in thread Thread-3 (_pin_memory_loop):
Traceback (most recent call last):
  File "/home/srs-9/.pyenv/versions/3.13.0/lib/python3.13/threading.py", line 1041, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "/home/srs-9/.pyenv/versions/3.13.0/lib/python3.13/threading.py", line 992, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/srs-9/.virtualenvs/monai/lib/python3.13/site-packages/torch/utils/data/_utils/pin_me

KeyboardInterrupt: 

In [17]:
import yaml
train_home = Path("/home/srs-9/Projects/prl_project/training/roi_train2")
train_root = Path("/media/smbshare/srs-9/prl_project/training/roi_train2")
train_name = "tmp2"
work_dir = train_root / train_name
work_dir.mkdir(exist_ok=True)

datalist_dst = work_dir/"datalist.json"
shutil.copy2(train_home/"datalist_flair.phase_xy25_z2.json", datalist_dst)

input_cfg = {
    "modality": "mri",
    "class_names": ["lesion", "rim"],
    "algos": ["swinunetr"],
    "work_dir": str(work_dir),
    "datalist": str(datalist_dst),
    "dataroot": str(DATA_ROOT),
    "analyze": True,
    "algo_gen": True,
    "train": False,
    "ensemble": False
}

input_overrides = "/home/srs-9/Projects/prl_project/training/full_brain/input_overrides.yaml"

with open("/media/smbshare/srs-9/prl_project/training/roi_train2/stage7_expand/run1/monai_config.json", 'r') as f:
    input_overrides = json.load(f)
input_overrides = input_overrides['train_param']
input_overrides['datalist'] = str(datalist_dst)
input_overrides['dataroot'] = str(DATA_ROOT)
input_overrides['work_dir'] = str(work_dir)
input_overrides['modality'] = "mri"
input_overrides['algos'] = ["segresnet"]

inp_override_path = work_dir/"input_overrides.yaml"
with open(inp_override_path, 'w') as f:
    yaml.dump(input_overrides, f)
runner = AutoRunner(input=str(inp_override_path), train=False, infer=False, analyze=True, algo_gen=True, ensemble=False)
runner.run()

Loading input config /media/smbshare/srs-9/prl_project/training/roi_train2/tmp2/input_overrides.yaml
AutoRunner using work directory /media/smbshare/srs-9/prl_project/training/roi_train2/tmp2
Found num_fold 5 based on the input datalist /media/smbshare/srs-9/prl_project/training/roi_train2/tmp2/datalist.json.
Setting num_fold 5 based on the input datalist /media/smbshare/srs-9/prl_project/training/roi_train2/tmp2/datalist.json.
Using user defined command running prefix , will override other settings
Running data analysis...


2026-03-30 19:04:52,206 - INFO - Found 1 GPUs for data analyzing!


monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  0%|          | 0/794 [00:00<?, ?it/s]

100%|██████████| 794/794 [00:10<00:00, 76.80it/s]


2026-03-30 19:05:02,947 - INFO - Data spacing is not completely uniform. MONAI transforms may provide unexpected result
2026-03-30 19:05:02,947 - INFO - Writing data stats to /media/smbshare/srs-9/prl_project/training/roi_train2/tmp2/datastats.yaml.
2026-03-30 19:05:02,980 - INFO - Writing by-case data stats to /media/smbshare/srs-9/prl_project/training/roi_train2/tmp2/datastats_by_case.yaml, this may take a while.
2026-03-30 19:05:05,642 - INFO - BundleGen from https://github.com/Project-MONAI/research-contributions/releases/download/algo_templates/21ed8e5.tar.gz


algo_templates.tar.gz: 104kB [00:00, 572kB/s]                              

2026-03-30 19:05:06,006 - INFO - Downloaded: /tmp/tmp43omg9ch/algo_templates.tar.gz
2026-03-30 19:05:06,006 - INFO - Expected md5 is None, skip md5 check for file /tmp/tmp43omg9ch/algo_templates.tar.gz.
2026-03-30 19:05:06,008 - INFO - Writing into directory: /media/smbshare/srs-9/prl_project/training/roi_train2/tmp2.


2026-03-30 19:05:07,406 - INFO - Generated:/media/smbshare/srs-9/prl_project/training/roi_train2/tmp2/segresnet_0
2026-03-30 19:05:07,822 - INFO - Generated:/media/smbshare/srs-9/prl_project/training/roi_train2/tmp2/segresnet_1
2026-03-30 19:05:08,224 - INFO - Generated:/media/smbshare/srs-9/prl_project/training/roi_train2/tmp2/segresnet_2
2026-03-30 19:05:08,645 - INFO - Generated:/media/smbshare/srs-9/prl_project/training/roi_train2/tmp2/segresnet_3
2026-03-30 19:05:09,044 - INFO - Generated:/media/smbshare/srs-9/prl_project/training/roi_train2/tmp2/segresnet_4


Skipping algorithm training...
Auto3Dseg pipeline is completed successfully.
